# Chemical Expansion Tensor Visualization - Fixed Version

**Material:** Ni → NiO Phase Transformation  
**Application:** SOFC Ni-YSZ Redox Eigenstrain Analysis  
**Author:** georgegershom  
**Date:** 2026-02-01  

## Fix Applied
Fixed `add_panel_label()` function to properly handle 3D axes by using `text2D()` instead of `text()` for Axes3D objects.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from dataclasses import dataclass

# Configuration
@dataclass
class FigureConfig:
    """Configuration for figure styling"""
    DPI: int = 300
    FIGURE_WIDTH: float = 16
    FIGURE_HEIGHT: float = 10
    TITLE_SIZE: int = 14
    LABEL_SIZE: int = 12
    TICK_SIZE: int = 10
    LEGEND_SIZE: int = 10
    LINE_WIDTH: float = 2.0
    
    COLORS: dict = None
    
    def __post_init__(self):
        if self.COLORS is None:
            self.COLORS = {
                'strain_11': '#E63946',  # Red
                'strain_22': '#2A9D8F',  # Teal
                'strain_33': '#F77F00',  # Orange
                'volumetric': '#264653', # Dark blue
                'grid': '#CCCCCC',
                'text_dark': '#1A1A1A',
                'text_light': '#666666'
            }

cfg = FigureConfig()

In [ ]:
# FIXED: add_panel_label function now handles 3D axes correctly
def add_panel_label(ax, label, cfg, x=-0.08, y=1.06):
    """
    Add professional panel label
    
    FIXED: Now handles both 2D and 3D axes correctly.
    For 3D axes (Axes3D), uses text2D() which accepts 2D coordinates.
    For 2D axes, uses regular text() method.
    """
    # Check if this is a 3D axis
    if hasattr(ax, 'text2D'):
        # For 3D axes, use text2D which works with 2D coordinates
        ax.text2D(x, y, f'({label})', 
                  transform=ax.transAxes,
                  fontsize=cfg.TITLE_SIZE + 2, 
                  fontweight='bold',
                  color=cfg.COLORS['text_dark'],
                  verticalalignment='top', 
                  horizontalalignment='left')
    else:
        # For 2D axes, use regular text
        ax.text(x, y, f'({label})', 
                transform=ax.transAxes,
                fontsize=cfg.TITLE_SIZE + 2, 
                fontweight='bold',
                color=cfg.COLORS['text_dark'],
                verticalalignment='top', 
                horizontalalignment='left')

In [ ]:
# Generate sample data
def generate_sample_data():
    """Generate sample chemical expansion tensor data"""
    temperatures = np.linspace(300, 1000, 100)  # K
    
    # Temperature-dependent strain components
    # Based on typical NiO expansion behavior
    strain_11 = 0.01 * (1 + 0.0005 * (temperatures - 300))
    strain_22 = 0.01 * (1 + 0.0005 * (temperatures - 300))
    strain_33 = 0.012 * (1 + 0.0004 * (temperatures - 300))
    
    # Volumetric strain
    volumetric = strain_11 + strain_22 + strain_33
    
    # Strain rate (derivative)
    strain_rate = np.gradient(volumetric, temperatures)
    
    return {
        'temperatures': temperatures,
        'strain_11': strain_11,
        'strain_22': strain_22,
        'strain_33': strain_33,
        'volumetric': volumetric,
        'strain_rate': strain_rate
    }

data = generate_sample_data()

In [ ]:
# Panel creation functions
def create_strain_components_panel(ax, cfg, data):
    """Panel (a): Individual strain components vs temperature"""
    ax.plot(data['temperatures'], data['strain_11'] * 100, 
            color=cfg.COLORS['strain_11'], linewidth=cfg.LINE_WIDTH,
            label=r'$\varepsilon_{11}$', marker='o', markersize=3, markevery=10)
    ax.plot(data['temperatures'], data['strain_22'] * 100,
            color=cfg.COLORS['strain_22'], linewidth=cfg.LINE_WIDTH,
            label=r'$\varepsilon_{22}$', marker='s', markersize=3, markevery=10)
    ax.plot(data['temperatures'], data['strain_33'] * 100,
            color=cfg.COLORS['strain_33'], linewidth=cfg.LINE_WIDTH,
            label=r'$\varepsilon_{33}$', marker='^', markersize=3, markevery=10)
    
    ax.set_xlabel('Temperature (K)', fontsize=cfg.LABEL_SIZE, fontweight='bold')
    ax.set_ylabel('Strain (%)', fontsize=cfg.LABEL_SIZE, fontweight='bold')
    ax.set_title('Chemical Expansion Strain Components', 
                 fontsize=cfg.TITLE_SIZE, fontweight='bold', pad=15)
    ax.legend(fontsize=cfg.LEGEND_SIZE, frameon=True, shadow=True)
    ax.grid(True, alpha=0.3, linestyle='--', color=cfg.COLORS['grid'])
    ax.tick_params(labelsize=cfg.TICK_SIZE)


def create_volumetric_panel(ax, cfg, data):
    """Panel (b): Volumetric expansion vs temperature"""
    ax.plot(data['temperatures'], data['volumetric'] * 100,
            color=cfg.COLORS['volumetric'], linewidth=cfg.LINE_WIDTH,
            label=r'$\varepsilon_v = \varepsilon_{11} + \varepsilon_{22} + \varepsilon_{33}$',
            marker='o', markersize=4, markevery=10)
    
    ax.fill_between(data['temperatures'], 0, data['volumetric'] * 100,
                     alpha=0.2, color=cfg.COLORS['volumetric'])
    
    ax.set_xlabel('Temperature (K)', fontsize=cfg.LABEL_SIZE, fontweight='bold')
    ax.set_ylabel('Volumetric Strain (%)', fontsize=cfg.LABEL_SIZE, fontweight='bold')
    ax.set_title('Volumetric Expansion', 
                 fontsize=cfg.TITLE_SIZE, fontweight='bold', pad=15)
    ax.legend(fontsize=cfg.LEGEND_SIZE - 1, frameon=True, shadow=True, loc='upper left')
    ax.grid(True, alpha=0.3, linestyle='--', color=cfg.COLORS['grid'])
    ax.tick_params(labelsize=cfg.TICK_SIZE)


def create_tensor_3d_panel(ax, cfg, data):
    """Panel (c): 3D tensor representation at specific temperature"""
    # Use mid-temperature data
    mid_idx = len(data['temperatures']) // 2
    
    # Create ellipsoid representing strain tensor
    u = np.linspace(0, 2 * np.pi, 50)
    v = np.linspace(0, np.pi, 50)
    
    # Scale by strain values
    a = 1 + data['strain_11'][mid_idx] * 50
    b = 1 + data['strain_22'][mid_idx] * 50
    c = 1 + data['strain_33'][mid_idx] * 50
    
    x = a * np.outer(np.cos(u), np.sin(v))
    y = b * np.outer(np.sin(u), np.sin(v))
    z = c * np.outer(np.ones(np.size(u)), np.cos(v))
    
    # Plot surface
    surf = ax.plot_surface(x, y, z, cmap=cm.coolwarm, 
                           alpha=0.8, linewidth=0, antialiased=True)
    
    # Add principal axes
    arrow_props = dict(mutation_scale=20, lw=2, arrowstyle='->', color='black')
    ax.quiver(0, 0, 0, a, 0, 0, color=cfg.COLORS['strain_11'], 
              arrow_length_ratio=0.2, linewidth=3, alpha=0.9)
    ax.quiver(0, 0, 0, 0, b, 0, color=cfg.COLORS['strain_22'],
              arrow_length_ratio=0.2, linewidth=3, alpha=0.9)
    ax.quiver(0, 0, 0, 0, 0, c, color=cfg.COLORS['strain_33'],
              arrow_length_ratio=0.2, linewidth=3, alpha=0.9)
    
    ax.set_xlabel(r'$\varepsilon_{11}$', fontsize=cfg.LABEL_SIZE, fontweight='bold')
    ax.set_ylabel(r'$\varepsilon_{22}$', fontsize=cfg.LABEL_SIZE, fontweight='bold')
    ax.set_zlabel(r'$\varepsilon_{33}$', fontsize=cfg.LABEL_SIZE, fontweight='bold')
    ax.set_title('3D Strain Tensor Representation',
                 fontsize=cfg.TITLE_SIZE, fontweight='bold', pad=20)
    ax.tick_params(labelsize=cfg.TICK_SIZE - 2)
    
    # Set viewing angle
    ax.view_init(elev=20, azim=45)


def create_strain_rate_panel(ax, cfg, data):
    """Panel (d): Strain rate vs temperature"""
    ax.plot(data['temperatures'], data['strain_rate'] * 100,
            color=cfg.COLORS['volumetric'], linewidth=cfg.LINE_WIDTH,
            label=r'$\frac{d\varepsilon_v}{dT}$',
            marker='o', markersize=3, markevery=10)
    
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    
    ax.set_xlabel('Temperature (K)', fontsize=cfg.LABEL_SIZE, fontweight='bold')
    ax.set_ylabel('Strain Rate (%/K)', fontsize=cfg.LABEL_SIZE, fontweight='bold')
    ax.set_title('Thermal Expansion Rate',
                 fontsize=cfg.TITLE_SIZE, fontweight='bold', pad=15)
    ax.legend(fontsize=cfg.LEGEND_SIZE, frameon=True, shadow=True)
    ax.grid(True, alpha=0.3, linestyle='--', color=cfg.COLORS['grid'])
    ax.tick_params(labelsize=cfg.TICK_SIZE)

In [ ]:
# Main figure creation function
def create_figure1():
    """Create complete Figure 1 with all panels"""
    fig = plt.figure(figsize=(cfg.FIGURE_WIDTH, cfg.FIGURE_HEIGHT), dpi=cfg.DPI)
    
    # Create grid layout
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.35,
                         left=0.08, right=0.95, top=0.93, bottom=0.08)
    
    # Create subplots
    ax_strain = fig.add_subplot(gs[0, 0])
    ax_vol = fig.add_subplot(gs[0, 1])
    ax_3d = fig.add_subplot(gs[1, 0], projection='3d')
    ax_rate = fig.add_subplot(gs[1, 1])
    
    # Generate panels
    print("Generating Panel (a): Strain Components...")
    create_strain_components_panel(ax_strain, cfg, data)
    add_panel_label(ax_strain, 'a', cfg)
    
    print("Generating Panel (b): Volumetric Expansion...")
    create_volumetric_panel(ax_vol, cfg, data)
    add_panel_label(ax_vol, 'b', cfg)
    
    print("Generating Panel (c): 3D Tensor...")
    create_tensor_3d_panel(ax_3d, cfg, data)
    add_panel_label(ax_3d, 'c', cfg, x=-0.05, y=1.02)  # FIXED: Now works with 3D axes
    
    print("Generating Panel (d): Strain Rate...")
    create_strain_rate_panel(ax_rate, cfg, data)
    add_panel_label(ax_rate, 'd', cfg)
    
    # Add figure title
    fig.suptitle('Chemical Expansion Tensor: Ni → NiO Phase Transformation',
                 fontsize=cfg.TITLE_SIZE + 4, fontweight='bold', y=0.98)
    
    return fig

In [ ]:
# Generate the figure
print("=" * 65)
print("FIGURE 1: CHEMICAL EXPANSION TENSOR INPUT")
print("=" * 65)
print("Material:     Ni → NiO Phase Transformation")
print("Application:  SOFC Ni-YSZ Redox Eigenstrain Analysis")
print("Model:        Temperature-Dependent Tensor")
print("Author:       georgegershom")
print("Date:         2026-02-01")
print("=" * 65 + "\n")

fig = create_figure1()

print("\n✨ Figure generation complete!\n")
plt.show()

In [ ]:
# Optional: Save the figure
# fig.savefig('chemical_expansion_tensor_figure1.png', dpi=300, bbox_inches='tight')
# fig.savefig('chemical_expansion_tensor_figure1.pdf', bbox_inches='tight')
# print("Figure saved successfully!")